# DroneAId — YOLO11s-seg Training on RescueNet (Kaggle GPU)

**Purpose:** Train YOLO11s-seg for terrain/damage semantic segmentation on RescueNet.

**Classes (8):** Water, Building (No/Minor/Major Damage, Total Destruction), Road-Clear, Road-Blocked, Vehicle

**Constraint budget:** Detection model = 18.1 MB ONNX → remaining 31.9 MB for this model

## 1. Environment check

In [21]:
!pip install -q ultralytics==8.4.37 onnx onnxruntime
import torch, platform, os, json
from pathlib import Path

print('torch        :', torch.__version__)
print('cuda avail   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name     :', torch.cuda.get_device_name(0))
    print('gpu memory   :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')
print('python       :', platform.python_version())

assert torch.cuda.is_available(), 'GPU required'

torch        : 2.9.0+cu126
cuda avail   : True
gpu name     : Tesla P100-PCIE-16GB
gpu memory   : 15.89 GB
python       : 3.12.12


## 2. Dataset verification

In [22]:
DST = Path('/kaggle/working/rescuenet_yolo')
data_yaml = DST / 'data.yaml'

assert DST.exists(), f'Dataset not found at {DST}. Run conversion first.'
assert data_yaml.exists(), 'data.yaml not found'

for split in ('train', 'val', 'test'):
    imgs = list((DST / split / 'images').glob('*'))
    lbls = list((DST / split / 'labels').glob('*.txt'))
    print(f'{split}: {len(imgs)} images, {len(lbls)} labels')

print()
print(data_yaml.read_text())

train: 720 images, 720 labels
val: 449 images, 449 labels
test: 450 images, 450 labels

path: /kaggle/working/rescuenet_yolo
train: train/images
val: val/images
test: test/images
nc: 8
names: ['water', 'building-no-damage', 'building-minor-damage', 'building-major-damage', 'building-total-destruction', 'road-clear', 'road-blocked', 'vehicle']



## 3. Training

| Key | Value | Reason |
|---|---|---|
| model | yolo11s-seg.pt | Matches detection model family |
| imgsz | 640 | Terrain features are large, 640 sufficient |
| epochs | 10 | Deadline constraint |
| patience | 5 | Early stop |
| batch | -1 | Auto-batch |
| save_period | 5 | Checkpoint every 5 epochs |

In [23]:
from ultralytics import YOLO

WORK = Path('/kaggle/working')
RUN_NAME = 'yolov12s_seg_rescuenet_v1'
PROJECT = str(WORK / 'runs')

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=10,
    patience=5,
    batch=-1,
    optimizer='auto',
    cos_lr=True,
    amp=True,
    cache='disk',
    workers=4,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    seed=42,
    verbose=True,
    save=True,
    save_period=5,
    plots=True,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rescuenet_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov12s_seg_rescuenet_v1, nbs=64, nms=False, opset=None, optimize=False, optim

## 4. Validation on test split

In [6]:
# SKIP validation (OOM + too slow on CPU) — go straight to export
import json
from pathlib import Path
from ultralytics import YOLO

WORK = Path("/kaggle/working")
RUN_NAME = "yolov12s_seg_rescuenet_v1"
PROJECT = str(WORK / "runs")

BEST_PT = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
assert BEST_PT.exists(), f'best weight not found: {BEST_PT}'
print('best checkpoint size (MB):', round(BEST_PT.stat().st_size / 1024**2, 3))
best_model = YOLO(str(BEST_PT))
print('Model loaded OK')

best checkpoint size (MB): 19.549
Model loaded OK


## 5. Export to ONNX

In [7]:
# FP16 export to fit C-A1 budget
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    opset=13,
    dynamic=False,
    simplify=True,
    half=True,
)
onnx_size = round(Path(onnx_path).stat().st_size / 1024**2, 3)
print(f'ONNX FP16: {onnx_path} ({onnx_size} MB)')

det_size = 18.093
total = det_size + onnx_size
print(f'Total pipeline: {det_size} + {onnx_size} = {total:.3f} MB')
print(f'C-A1 (total <= 50 MB): {"PASS" if total <= 50 else "FAIL"}')

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11s-seg summary (fused): 114 layers, 10,069,912 parameters, 0 gradients, 32.8 GFLOPs

PyTorch: starting from '/kaggle/working/runs/yolov12s_seg_rescuenet_v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 44, 8400), (1, 32, 160, 160)) (19.5 MB)

ONNX: starting export with onnx 1.20.1 opset 13...
ONNX: slimming with onnxslim 0.1.91...
ONNX: converting to FP16...
ONNX: export success ✅ 3.3s, saved as '/kaggle/working/runs/yolov12s_seg_rescuenet_v1/weights/best.onnx' (19.4 MB)

Export complete (4.3s)
Results saved to /kaggle/working/runs/yolov12s_seg_rescuenet_v1/weights
Predict:         yolo predict task=segment model=/kaggle/working/runs/yolov12s_seg_rescuenet_v1/weights/best.onnx imgsz=640 half
Validate:        yolo val task=segment model=/kaggle/working/runs/yolov12s_seg_rescuenet_v1/weights/best.onnx imgsz=640 data=/kaggle/working/rescuenet_yolo/data.yaml half 
Vi

## 6. Summary

In [8]:
summary = {
    'run_name': RUN_NAME,
    'model': 'YOLO11s-seg',
    'dataset': 'RescueNet',
    'imgsz': 640,
    'train_images': 720,
    'val_images': 449,
    'test_images': 450,
    'classes': 8,
    'best_checkpoint_mb': round(BEST_PT.stat().st_size / 1024**2, 3),
    'onnx_mb': onnx_size,
    'constraint_budget': {
        'det_model_mb': det_size,
        'seg_model_mb': onnx_size,
        'total_mb': round(total, 3),
        'c_a1_limit_mb': 50.0,
        'c_a1_pass': total <= 50.0,
    },
}
summary_path = WORK / f'{RUN_NAME}_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "run_name": "yolov12s_seg_rescuenet_v1",
  "model": "YOLO11s-seg",
  "dataset": "RescueNet",
  "imgsz": 640,
  "train_images": 720,
  "val_images": 449,
  "test_images": 450,
  "classes": 8,
  "best_checkpoint_mb": 19.549,
  "onnx_mb": 19.401,
  "constraint_budget": {
    "det_model_mb": 18.093,
    "seg_model_mb": 19.401,
    "total_mb": 37.494,
    "c_a1_limit_mb": 50.0,
    "c_a1_pass": true
  }
}
